# Экспорты цифрового мастер-плана

Независимая нарезка BlocksNet, графы walk/drive/transit, сетевые зоны школ (15 мин), детсадов (7 мин) и поликлиник (15 мин), спрос, загрузка и связи ObjectNat. Канонические данные не заменяются. Интернет не нужен. Запустить из корня репозитория либо notebooks/. Каждый расчёт запускается в собственном окружении.

ObjectNat отбрасывает дробный спрос каждого дома: сравнение с непрерывным нормативным спросом обязательно; это методическое ограничение, а не дефицит вместимости.

In [1]:
from pathlib import Path
import os, sys, subprocess, json, pickle
import geopandas as gpd
import pandas as pd
ROOT=Path.cwd() if (Path.cwd()/'scripts').is_dir() else Path.cwd().parent
assert (ROOT/'scripts/analytical_exports.py').exists()
env=dict(os.environ, NUMBA_CACHE_DIR='/tmp/primorsk-numba', MPLCONFIGDIR='/tmp/primorsk-mpl')
print(ROOT)

/home/mfthesun/Projects/primorsk


In [2]:
# Полное воспроизведение обоих методов в совместимых окружениях.
for python, script in [('.venv-blocksnet/bin/python','scripts/blocks_generation.py'),('.venv-objectnat/bin/python','scripts/analytical_exports.py')]:
    completed=subprocess.run([str(ROOT/python),script],cwd=ROOT,env=env,check=True,capture_output=True,text=True)
    print(script, 'успешно; вывод строк:',len(completed.stdout.splitlines()))

scripts/blocks_generation.py успешно; вывод строк: 1


scripts/analytical_exports.py успешно; вывод строк: 3


In [3]:
rows=[]
for tag in ['today','s1','s2']:
    metrics=json.loads((ROOT/f'data/processed/analytical_export/{tag}/metrics.json').read_text())
    for mode in ['walk','drive','transit']:
        folder=ROOT/f'data/processed/analytical_export/{tag}'
        with (folder/f'graph_{mode}.pkl').open('rb') as stream: graph=pickle.load(stream)
        edges=gpd.read_file(folder/f'graph_{mode}.edges.geojson')
        assert edges.crs.to_epsg()==4326 and (edges.time_min>0).all() and (edges.length_meter>0).all()
        assert len(edges)==graph.number_of_edges()
        rows.append(dict(scenario=tag,mode=mode,**metrics['graphs'][mode]))
pd.DataFrame(rows)

,scenario,mode,nodes,edges,time_min_min,length_meter_sum
0,today,walk,8596,10172,0.012046,1.410117e+06
1,today,drive,7974,9086,0.010000,1.223661e+06
2,today,transit,8596,10172,0.012046,1.410117e+06
3,s1,walk,8881,10456,0.012046,1.410104e+06
4,s1,drive,7974,9086,0.010000,1.223661e+06
5,s1,transit,8881,10462,0.012046,1.439418e+06
6,s2,walk,8881,10456,0.012046,1.410104e+06
7,s2,drive,7974,9086,0.010000,1.223661e+06
8,s2,transit,8881,10462,0.012046,1.439418e+06


In [4]:
coverage_rows=[]
for tag in ['today','s1','s2']:
    folder=ROOT/f'data/processed/analytical_export/{tag}'
    for svc,threshold in [('school',15),('kindergarten',7),('polyclinic',15)]:
        zones=gpd.read_file(folder/f'coverage_{svc}_walk.geojson')
        assert zones.geometry.is_valid.all() and zones.geom_type.isin(['Polygon','MultiPolygon']).all()
        assert set(zones.threshold_min)=={threshold} and set(zones.scenario)=={tag}
        coverage_rows.append({'scenario':tag,'service':svc,'zones':len(zones),'union_area_ha':zones.to_crs(32635).union_all().area/1e4})
pd.DataFrame(coverage_rows)

,scenario,service,zones,union_area_ha
0,today,school,6,807.651851
1,today,kindergarten,4,150.385447
2,today,polyclinic,5,764.501253
3,s1,school,6,808.240786
4,s1,kindergarten,4,150.700309
5,s1,polyclinic,5,765.507840
6,s2,school,6,808.240786
7,s2,kindergarten,5,162.564804
8,s2,polyclinic,5,765.507840


In [5]:
rows=[]
for tag in ['today','s1','s2']:
    folder=ROOT/f'data/processed/analytical_export/{tag}'
    metrics=json.loads((folder/'metrics.json').read_text())
    for svc in ['school','kindergarten']:
        b=gpd.read_file(folder/f'provision_{svc}_buildings.geojson')
        s=gpd.read_file(folder/f'provision_{svc}_services.geojson')
        links=gpd.read_file(folder/f'provision_{svc}_links.geojson')
        flow=pd.read_pickle(folder/f'provision_{svc}_flow.pkl')
        assert b.object_id.is_unique and s.object_id.is_unique
        assert set(links.building_id)<=set(b.object_id) and set(links.service_object_id)<=set(s.object_id)
        assert abs(links.allocated.sum()-flow.sparse.to_coo().data.sum())<1e-8
        assert abs(links.allocated.sum()-(b.demand.sum()-b.demand_left.sum()))<1e-8
        assert abs(links.allocated.sum()-(s.capacity.sum()-s.capacity_left.sum()))<1e-8
        assert (links.within_threshold==(links.time_min<=links.threshold_min)).all()
        rows.append(dict(scenario=tag,service=svc,**metrics['provision'][svc]))
pd.DataFrame(rows)[['scenario','service','source_demand','library_integer_demand','allocated','supplied_within','supplied_outside','source_capacity']]

,scenario,service,source_demand,library_integer_demand,allocated,supplied_within,supplied_outside,source_capacity
0,today,school,1025.079,429,409.0,384,25,1847.0
1,today,kindergarten,680.676,256,230.0,202,28,470.0
2,s1,school,1031.904,435,409.0,384,25,1847.0
3,s1,kindergarten,685.251,260,230.0,202,28,470.0
4,s2,school,1031.904,435,409.0,384,25,1847.0
5,s2,kindergarten,685.251,260,231.0,202,29,630.0


In [6]:
metrics=json.loads((ROOT/'reports/blocks_generation.json').read_text())
assert metrics['canonical_count']==526
assert metrics['generated_count']>0
assert 0<=metrics['canonical_area_covered_share']<=1
print(json.dumps({k:v for k,v in metrics.items() if k not in ['inputs','method']},ensure_ascii=False,indent=2))

{
  "generated_count": 361,
  "canonical_count": 526,
  "generated_area_ha": 66481.39838275117,
  "canonical_area_ha": 64959.25653051348,
  "boundary_area_ha": 335997.2041361205,
  "canonical_area_covered_share": 0.999974325670004,
  "generated_area_in_canonical_share": 0.9770791578592681,
  "generated_boundary_coverage_share": 0.19786295113282787,
  "intersection_over_union": 0.9770546469775591,
  "generated_internal_overlap_m2": 4.410743713378906e-06,
  "canonical_split_count": 15,
  "generated_merge_count": 36,
  "overlap_pairs": 548,
  "overlap_pair_min_area_m2": 1
}


Зоны охвата построены get_graph_coverage: Dijkstra, Voronoi и буфер дорожной сети 60 м; вместимость не ограничивает охват. Связи дом—сервис показывают фактическую flow библиотеки, прямые линии не являются маршрутами. Нулевая вместимость поликлиник означает отсутствие сведений, поэтому их обеспеченность по мощности не рассчитывается. Прикрепление к ближайшему узлу не учитывает подход к сети. Независимая нарезка использует существующие дороги/ж.д., реки и водные полигоны, без дополнительной ручной корректировки, classifier или snapping. Исходные 526 кварталов и новый эксперимент могут иметь разную детализацию; сравнение площадей и пересечений — в reports/blocks_generation.md.